# LangGraph와 AgentCore Memory Tool(단기 메모리)

## 소개
이 Notebook에서는 LangGraph framework를 사용하여 Amazon Bedrock AgentCore Memory 기능을 대화형 AI Agent와 통합하는 방법을 살펴봅니다. 하나의 대화 세션 안에서 **단기 메모리**를 유지하여 명시적인 컨텍스트 관리 없이 Agent가 이전 대화 내용을 기억하도록 하는 데 중점을 둡니다.


## 튜토리얼 세부 정보

| 정보         | 세부 정보                                                                          |
|:--------------------|:---------------------------------------------------------------------------------|
| 튜토리얼 유형       | 단기 대화                                                        |
| Agent 사용 사례       | 개인 피트니스                                                                 |
| Agentic Framework   | Langgraph                                                                        |
| LLM 모델           | Anthropic Claude Haiku 4.5                                                      |
| 튜토리얼 구성 요소 | AgentCore Short-term Memory, Langgraph, Memory retrieval via Tool                |
| 예제 난이도  | 초급                                                                         |

다음 내용을 학습합니다.
- 단기 메모리를 위한 AgentCore Memory 저장소 생성
- LangGraph로 구조화된 메모리 workflow를 사용하는 Agent 생성
- 대화 기록 검색용 Memory Tool 구현
- 단일 세션 내에서 컨텍스트 정보에 액세스하고 활용
- 효과적인 메모리 회상으로 대화 환경 개선


### 시나리오 배경

이 예제에서는 대화 중에 언급된 운동 세부 정보, 피트니스 목표, 신체적 제약, 운동 선호도를 기억할 수 있는 "**Personal Fitness Coach**"를 만듭니다. 이 Assistant를 통해 효과적인 단기 메모리 관리가 사용자에게 같은 정보를 반복해서 요구하지 않으면서 더 자연스럽고 개인화된 피트니스 코칭 환경을 제공하는 방법을 살펴봅니다.


## 아키텍처
<div style="text-align:left">
    <img src="images/architecture.png" width="65%" />
</div>

## 사전 요구 사항

- Python 3.10 이상
- 적절한 권한이 있는 AWS 계정
- AgentCore Memory에 적절한 권한이 있는 AWS IAM 역할
- Amazon Bedrock 모델에 대한 액세스

먼저 환경을 설정하겠습니다.

## 1단계: 환경 설정
Notebook 실행에 필요한 모든 라이브러리를 가져오고 client를 정의합니다.

In [ ]:
!pip install -qr requirements.txt

In [ ]:
import logging
from datetime import datetime

Amazon Bedrock 모델 및 AgentCore에 적절한 권한이 있는 리전과 역할을 정의합니다.

In [ ]:
import os

region = os.getenv("AWS_REGION", "us-west-2")

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
logger = logging.getLogger("agentcore-memory")

### 통합 작동 방식

LangGraph와 AgentCore Memory의 통합은 다음과 같이 작동합니다.

1. AgentCore Memory를 사용하여 대화를 단기 메모리에 저장
2. LangGraph의 구조화된 workflow로 메모리 작업 관리

이 접근 방식은 메모리 관리와 추론을 분리하여 더 깔끔하고 유지 관리하기 쉬운 Agent 아키텍처를 구현합니다.

## 2단계: Memory 생성
이 섹션에서는 AgentCore Memory SDK를 사용하여 Memory 저장소를 생성합니다. 이 저장소를 통해 Agent가 대화 정보를 유지할 수 있습니다.

In [ ]:
from bedrock_agentcore.memory import MemoryClient
from botocore.exceptions import ClientError

In [ ]:
client = MemoryClient(region_name=region)
memory_name = "FitnessCoach"
memory_id = None

In [ ]:
try:
    print("Creating Memory...")
    # Memory 리소스 생성
    memory = client.create_memory_and_wait(
        name=memory_name,  # 이 계정의 모든 Memory에서 고유한 이름
        description="Fitness Coach Agent",  # 사람이 읽을 수 있는 설명
        strategies=[],  # 단기 메모리에는 Memory strategy를 사용하지 않음
        event_expiry_days=7,  # Memory는 7일 후 만료
        max_wait=300,  # Memory 생성을 기다리는 최대 시간(5분)
        poll_interval=10,  # 10초마다 상태 확인
    )

    # Memory ID 추출 및 출력
    memory_id = memory["id"]
    logger.info(f"Memory created successfully with ID: {memory_id}")
except ClientError as e:
    if e.response["Error"]["Code"] == "ValidationException" and "already exists" in str(e):
        # Memory가 이미 존재하면 ID 검색
        memories = client.list_memories()
        memory_id = next((m["id"] for m in memories if m["id"].startswith(memory_name)), None)
        logger.info(f"Memory already exists. Using existing memory ID: {memory_id}")
except Exception as e:
    # Memory 생성 중 발생한 오류 처리
    logger.info(f"❌ ERROR: {e}")
    import traceback

    traceback.print_exc()
    # 오류 발생 시 정리 - 일부 생성된 Memory 삭제
    if memory_id:
        try:
            client.delete_memory_and_wait(memory_id=memory_id)
            logger.info(f"Cleaned up memory: {memory_id}")
        except Exception as cleanup_error:
            logger.info(f"Failed to clean up memory: {cleanup_error}")

## 3단계: LangGraph Agent 생성
LangGraph로 Agent를 만드는 데 필요한 모든 라이브러리를 가져옵니다.

In [ ]:
from langgraph.graph import StateGraph, MessagesState
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_aws import ChatBedrock

### LangGraph Agent 구현

이제 Memory Tool을 통합하여 LangGraph Agent를 생성합니다.

In [ ]:
def create_agent(client, memory_id, actor_id, session_id):
    """LangGraph 에이전트를 생성하고 구성합니다."""

    # LLM 초기화(필요에 따라 모델과 parameter 조정)
    llm = ChatBedrock(
        model_id="global.anthropic.claude-haiku-4-5-20251001-v1:0",  # 또는 선호하는 모델
        model_kwargs={"temperature": 0.1},
    )

    @tool
    def list_events():
        """Tool used when needed to retrieve recent information"""
        events = client.list_events(
            memory_id=memory_id,
            actor_id=actor_id,
            session_id=session_id,
            max_results=10,
        )
        return events

    # 도구를 LLM에 바인딩
    tools = [list_events]
    llm_with_tools = llm.bind_tools(tools)

    # system message 설정
    system_message = """You are the Personal Fitness Coach, a sophisticated fitness guidance assistant.
                        PURPOSE:
                        - Help users develop workout routines based on their fitness goals
                        - Remember user's exercise preferences, limitations, and progress
                        - Provide personalized fitness recommendations and training plans
                        MEMORY CAPABILITIES:
                        - You have access to recent events with the list_events tool
                        """

    # Chatbot node 정의
    def chatbot(state: MessagesState):
        raw_messages = state["messages"]

        # 중복 또는 잘못된 배치를 방지하기 위해 기존 system message 제거
        non_system_messages = [msg for msg in raw_messages if not isinstance(msg, SystemMessage)]

        # SystemMessage가 항상 첫 번째가 되도록 설정
        messages = [SystemMessage(content=system_message)] + non_system_messages

        latest_user_message = next(
            (msg.content for msg in reversed(messages) if isinstance(msg, HumanMessage)),
            None,
        )

        # 도구가 바인딩된 모델에서 응답 가져오기
        response = llm_with_tools.invoke(messages)

        # 해당하는 경우 대화 저장
        if latest_user_message and response.content.strip():  # 응답에 내용이 있는지 확인
            conversation = [
                (latest_user_message, "USER"),
                (response.content, "ASSISTANT"),
            ]

            # 모든 메시지 텍스트가 비어 있지 않은지 검증
            if all(msg[0].strip() for msg in conversation):  # 빈 메시지가 없는지 확인
                try:
                    client.create_event(
                        memory_id=memory_id,
                        actor_id=actor_id,
                        session_id=session_id,
                        messages=conversation,
                    )
                except Exception as e:
                    print(f"Error saving conversation: {str(e)}")

        # 전체 메시지 기록에 응답 추가
        return {"messages": raw_messages + [response]}

    # Graph 생성
    graph_builder = StateGraph(MessagesState)

    # Node 추가
    graph_builder.add_node("chatbot", chatbot)
    graph_builder.add_node("tools", ToolNode(tools))

    # Edge 추가
    graph_builder.add_conditional_edges(
        "chatbot",
        tools_condition,
    )
    graph_builder.add_edge("tools", "chatbot")

    # Entry point 설정
    graph_builder.set_entry_point("chatbot")

    # graph compile 수행
    return graph_builder.compile()

### Agent 호출용 래퍼 생성

Agent를 호출하는 간단한 래퍼를 생성합니다.

In [ ]:
def langgraph_bedrock(payload, agent):
    """
    페이로드로 에이전트를 호출합니다.
    """
    user_input = payload.get("prompt")

    # LangGraph에서 요구하는 형식으로 입력 생성
    response = agent.invoke({"messages": [HumanMessage(content=user_input)]})

    # 최종 메시지 내용 추출
    return response["messages"][-1].content

## 4단계: LangGraph Agent 실행
이제 AgentCore Memory가 통합된 Agent를 실행할 수 있습니다.

In [ ]:
# 이 대화에 사용할 고유 actor 및 session ID 생성
actor_id = f"user-{datetime.now().strftime('%Y%m%d%H%M%S')}"
session_id = f"workout-{datetime.now().strftime('%Y%m%d%H%M%S')}"

In [ ]:
# AgentCore Memory가 통합된 Agent 생성
agent = create_agent(client, memory_id, actor_id, session_id)

#### 축하합니다! Agent가 준비되었습니다!

### Agent 테스트

Agent와 상호작용하여 메모리 기능을 테스트해 보겠습니다.

In [ ]:
response = langgraph_bedrock({"prompt": "Hello! This is my first day, I need a workout routine."}, agent)
print(f"Agent: {response}\n")

In [ ]:
response = langgraph_bedrock(
    {"prompt": "I want to build muscle, looking for a biceps routine. I have some lower back problems."},
    agent,
)
print(f"Agent: {response}\n")

In [ ]:
response = langgraph_bedrock({"prompt": "Can you give me three exercises with number of reps?"}, agent)
print(f"Agent: {response}\n")

### 메모리 지속성 테스트

AgentCore Memory 통합의 기능을 확인하기 위해 새 Agent 인스턴스를 만들고 이전 대화를 기억하는지 살펴보겠습니다.

In [ ]:
# 새 Agent 인스턴스 생성(새 세션 모의)
new_agent = create_agent(client, memory_id, actor_id, session_id)

# 새 Agent가 선호도를 기억하는지 테스트
response = langgraph_bedrock(
    {"prompt": "Hello again! Can you remind me about my last workout session?"},
    new_agent,
)

print("New Agent Session:\n")
print(f"Agent: {response}")

## 요약

이 Notebook에서는 다음 내용을 구현했습니다.

1. AI Agent용 AgentCore Memory 리소스를 생성하는 방법
2. 메모리가 통합된 LangGraph workflow 구축
3. 대화 기록 검색용 Memory Tool 구현
4. 필요할 때 Memory를 지능적으로 사용하는 Agent 생성
5. Agent 인스턴스 간 메모리 지속성 테스트

이 통합은 구조화된 workflow(LangGraph)와 강력한 메모리 시스템(AgentCore Memory)을 결합하여 더 지능적이고 컨텍스트를 인식하는 AI Agent를 만드는 방법을 보여 줍니다.

이 접근 방식은 Multi-Agent System, 추출 strategy를 사용하는 장기 메모리, 대화 컨텍스트 기반의 전문 메모리 검색 등 더 복잡한 사용 사례로 확장할 수 있습니다.

## 리소스 정리
이 Notebook에서 사용한 리소스를 정리하기 위해 Memory를 삭제합니다.

In [ ]:
# client.delete_memory_and_wait(memory_id = memory_id, max_wait = 300, poll_interval =10)